In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
base_dir = "/content/drive/MyDrive/SIB/dataset"
output_dir = "/content/drive/MyDrive/SIB/receipt-yolo-fix2"

In [3]:
import os

train_images = os.path.join(base_dir, "train/images")
train_labels = os.path.join(base_dir, "train/labels")

valid_images = os.path.join(base_dir, "valid/images")
valid_labels = os.path.join(base_dir, "valid/labels")

test_images = os.path.join(base_dir, "test/images")
test_labels = os.path.join(base_dir, "test/labels")

for path in [train_images, train_labels, valid_images, valid_labels, test_images, test_labels]:
    os.makedirs(path, exist_ok=True)

print("✅ Struktur folder siap di Google Drive")

✅ Struktur folder siap di Google Drive


In [4]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/SIB/dataset/train/_annotations.csv")  # ganti sesuai file kamu
df.head()

,filename,width,height,class,xmin,ymin,xmax,ymax
0,receipt_image_214_jpg.rf.a69fe454de88dab4b87ad...,640,640,Title,194,65,372,80
1,receipt_image_214_jpg.rf.a69fe454de88dab4b87ad...,640,640,Address,197,80,375,109
2,receipt_image_214_jpg.rf.a69fe454de88dab4b87ad...,640,640,Date,189,113,281,137
3,receipt_image_214_jpg.rf.a69fe454de88dab4b87ad...,640,640,Item,122,187,456,217
4,receipt_image_214_jpg.rf.a69fe454de88dab4b87ad...,640,640,TotalPrice,119,242,452,262


In [5]:
import yaml
import os

classes = sorted(df["class"].astype(str).unique())  # dari dataset kamu

data_yaml = {
    "train": train_images,
    "val": valid_images,
    "test": test_images,
    "nc": len(classes),
    "names": classes
}

yaml_path = os.path.join(output_dir, "data.yaml")

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("✅ data.yaml tersimpan di Google Drive!")
print("Lokasi:", yaml_path)

✅ data.yaml tersimpan di Google Drive!
Lokasi: /content/drive/MyDrive/SIB/receipt-yolo-fix2/data.yaml


In [6]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.4 MB/s eta 0:00:00


In [7]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data=yaml_path,
        epochs=50,
        imgsz=640,
        batch=8,
        name="receipt_model",
        box=7.5,
        cls=0.5,
        dfl=1.5,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/SIB/receipt-yolo-fix2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a1c0a9f5760>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

In [ ]:
import shutil

src = "/content/runs/detect/receipt_model"
dst = "/content/drive/MyDrive/SIB/receipt-yolo-fix2/receipt_model"

shutil.copytree(src, dst, dirs_exist_ok=True)

print("✅ Training result + best.pt sudah di Drive!")

In [ ]:
model.export(format="keras")